In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
import os

validated_root = "/Workspace/Users/edwinvictor73@gmail.com/FPL-transfer-research-assistant/src/Data/Validated_data/fpl_chips_test"

folders = [
    folder for folder in os.listdir(validated_root)
    if os.path.isdir(os.path.join(validated_root, folder))
]

# Sort timestamp folders: newest first
folders.sort(reverse=True)

if len(folders) < 2:
    raise Exception("Need at least two snapshots for CDC")

latest_folder = folders[0]
previous_folder = folders[1]

print("Latest snapshot:", latest_folder)
print("Previous snapshot:", previous_folder)

In [0]:
print("Latest folder contents:")
print(os.listdir(os.path.join(validated_root, latest_folder)))

print("Previous folder contents:")
print(os.listdir(os.path.join(validated_root, previous_folder)))

In [0]:
latest_path = os.path.join(
    validated_root,
    latest_folder,
    "fpl_chips_test.csv"
)

previous_path = os.path.join(
    validated_root,
    previous_folder,
    "fpl_chips_test.csv"
)

In [0]:
df_latest = spark.read.option("header", True).option(
    "inferSchema", True
).csv(latest_path)

df_previous = spark.read.option("header", True).option(
    "inferSchema", True
).csv(previous_path)

In [0]:
print("LATEST SNAPSHOT")
display(df_latest)

print("PREVIOUS SNAPSHOT")
display(df_previous)

In [0]:
from pyspark.sql.functions import col

primary_key = [
    "Chip_Name",
    "Start_Gameweek",
    "End_Gameweek"
]

# INSERT
df_inserts = df_latest.join(
    df_previous,
    on=primary_key,
    how="left_anti"
)

# DELETE candidates
df_deletes = df_previous.join(
    df_latest,
    on=primary_key,
    how="left_anti"
)

# UPDATE
df_updates = (
    df_latest.alias("latest")
    .join(
        df_previous.alias("previous"),
        on=primary_key,
        how="inner"
    )
    .filter(
        ~col("latest.chip_type").eqNullSafe(
            col("previous.chip_type")
        )
    )
)

In [0]:
print("INSERT RECORDS")
display(df_inserts)

print("UPDATE RECORDS")
display(df_updates)

print("DELETE CANDIDATES")
display(df_deletes)